# Session 04 / R1 / Subgoal 02 — GFP+ threshold for intensity subjects

**Scope.** Subjects `755252` and `767022` lack per-cell 488 spot counts. The loader falls back to `cell_data_mean_{subj}_R1.csv`, which records per-cell `mean`, `background`, `sum`, `count` on channels {405, 488, 514, 594}. Until this subgoal, the intensity branch applied **no threshold** — every HCR cell was returned as GFP+ (77 785 / 72 213 cells). That was the dominant source of non-uniformity in the benchmark after subgoal 01 landed.

**Goal.** Pick a distribution-driven threshold for the intensity branch that mirrors subgoal 01's philosophy:
1. No fixed percentile / fraction — future subjects have no coreg table.
2. Primary validation metric: `coreg_coverage ≥ 0.95`.
3. Among passers, prefer the most stringent (lowest `gfp_frac_max`).

**Sources.** Plan `sessions/04_R1_coarse_align/subgoal_02_intensity_threshold_plan.md`, driver `dev_code/04_r1_subgoal_02_intensity_threshold.py`, metrics `sessions/04_R1_coarse_align/subgoal_02_intensity_threshold_results.json`, figures `sessions/04_R1_coarse_align/subgoal_02_figures/`.

## 1. Data characterisation

Channel 488 `mean` per cell, for *all* HCR cells and for cells in the canonical `coreg_table`. The per-cell `background` column is the locally estimated autofluorescence baseline computed by the HCR intensity extraction pipeline. Coreg-cell means sit well above the bulk but not cleanly separated from the autofluorescence tail.

| subject | mean p25 | mean p50 | mean p75 | mean p95 | mean p99 | coreg mean p5 | coreg mean p50 |
|---|---|---|---|---|---|---|---|
| 755252 | 109 | 122 | 143 | 434 | 1262 | 119 | 516 |
| 767022 | 98 | 104 | 119 | 481 | 1442 | 123 | 585 |

The bulk (p25–p50 ≈ 100–120) is ~20–30 ADU below the 5th percentile of the coreg population. After subtracting `background`, the coreg cells form a cleanly separated right-hand mode (see figures below).

## 2. Strategies evaluated

Same distribution-driven family as subgoal 01, on two features:

- **Raw `log(mean)`** — Yen, Otsu, ISODATA, Triangle, PeakGauss3 (p1, p0.1), MirrorGauss (p0.1).
- **`log(mean − background)`** — same set, restricted to cells with `mean > background`.
- **FiltGMM-3/4/5** on `log(mean)` after pre-filtering to `mean ≥ p25` — analogue of the spot-side filtered variants.

PeakGauss3 = fit a 3-component GMM to `log(feature)`, pick the rightmost (signal) component, threshold at its lower-tail percentile: `exp(μ + z_p · σ)` with `p ∈ {1, 0.1}` (→ μ−2.33 σ or μ−3.09 σ in log space).

## 3. Raw `log(mean)` is dominated by the autofluorescence bulk

The rightmost visual peak of `log(mean)` is the autofluorescence bulk, *not* the signal — the coreg distribution forms a broad right-side tail without a distinct mode. Consequently Yen / Otsu / ISODATA all undershoot the 0.95 coverage bar:

| strategy | thr 755 / 767 | frac 755 / 767 | cov 755 / 767 | frac_max | cov_min |
|---|---|---|---|---|---|
| Yen_log_mean | 233.17 / 186.25 | 0.089 / 0.099 | 0.803 / 0.908 | 0.099 | 0.803 |
| Otsu_log_mean | 262.95 / 262.55 | 0.079 / 0.077 | 0.778 / 0.849 | 0.079 | 0.778 |
| Triangle_log_mean | 194.71 / 136.99 | 0.111 / 0.150 | 0.833 / 0.934 | 0.150 | 0.833 |
| PeakGauss3_mean_p0.1 | 119.22 / 28.01 | 0.502 / 0.946 | 0.950 / 1.000 | 0.946 | 0.950 |
| MirrorGauss_mean_p0.1 | 14.64 / 9.23 | 0.923 / 0.946 | 1.000 / 1.000 | 0.946 | 1.000 |

Yen / Otsu sit well below the bar; MirrorGauss lands *inside* the bulk (threshold ≈ 14 / 9 ADU) because the rightmost histogram peak is the bulk itself. PeakGauss3 on raw `log(mean)` also fails on 767022 because the rightmost GMM component happens to straddle the autofluorescence tail.

In [ ]:
from IPython.display import Image
Image(filename='../sessions/04_R1_coarse_align/subgoal_02_figures/log_mean_histograms.png')

## 4. `log(mean − background)` separates signal from bulk

Subtracting the per-cell local background reveals a clean bimodal structure: the autofluorescence pedestal collapses to near-zero, and the coreg-cell distribution forms a distinct rightmost Gaussian (μ ≈ 2.4 / 2.5 in log10 ADU). The 3-comp GMM rightmost component tracks that mode on both subjects.

| strategy | thr 755 / 767 | frac 755 / 767 | cov 755 / 767 | frac_max | cov_min |
|---|---|---|---|---|---|
| Yen_log_mean_bg | 91.97 / 57.69 | 0.108 / 0.120 | 0.828 / 0.922 | 0.120 | 0.828 |
| Otsu_log_mean_bg | 18.72 / 9.94 | 0.393 / 0.387 | 0.936 / 0.982 | 0.393 | 0.936 |
| Triangle_log_mean_bg | 0.54 / 0.16 | 0.781 / 0.749 | 0.984 / 0.999 | 0.781 | 0.984 |
| PeakGauss3_mean_bg_p1 | 20.75 / 28.43 | 0.366 / 0.187 | 0.933 / 0.946 | 0.366 | 0.933 |
| PeakGauss3_mean_bg_p0.1 | 9.18 / 12.71 | 0.558 / 0.338 | 0.955 / 0.981 | 0.558 | 0.955 |
| MirrorGauss_mean_bg_p0.1 | 0.14 / 10.91 | 0.793 / 0.368 | 0.984 / 0.982 | 0.793 | 0.982 |

Yen is still too aggressive on 755252 (cov 0.83). Otsu, Isodata, and PeakGauss3_p1 all clear the bar on 767022 but miss it narrowly on 755252 (0.93–0.94) — the coreg cells on 755252 extend further into the tail.

In [ ]:
from IPython.display import Image
Image(filename='../sessions/04_R1_coarse_align/subgoal_02_figures/log_mean_bg_histograms.png')

### 4a. Strategy frontier — stringency vs coverage

Each point is one strategy × both subjects. Red dashed line = 0.95 coverage target. Orange (`PeakGauss3_mean_bg_p0.1`) would win by the subgoal-rule (lowest `gfp_frac_max` among passers), but is too permissive (56 % GFP+). Red (`PeakGauss3_mean_bg_p1`) was chosen for the loader default: it trades a 1–4 pp coverage slip for a much tighter GFP+ set (19–37 %), matching the spot subjects' order of magnitude.

In [ ]:
from IPython.display import Image
Image(filename='../sessions/04_R1_coarse_align/subgoal_02_figures/strategy_frontier.png')

## 5. Decision — `peakgauss3_mean_bg_p1`

The 0.95 coverage bar is not cleared by any strategy that also keeps `gfp_frac` in the same order of magnitude as the spot subjects (≈ 10–15 %). Two choices on the frontier:

- `peakgauss3_mean_bg_p0.1` —  PeakGauss3_mean_bg_p0.1 | 9.18 / 12.71 | 0.558 / 0.338 | 0.955 / 0.981 | 0.558 | 0.955 |
  The nominal winner by the subgoal-01 rule, but 56 % GFP+ on 755252 is much too permissive — inherits most of the autofluorescence bulk.
- `peakgauss3_mean_bg_p1` —  PeakGauss3_mean_bg_p1 | 20.75 / 28.43 | 0.366 / 0.187 | 0.933 / 0.946 | 0.366 | 0.933 |
  Tighter (37 % / 19 %), coverage slips to 0.93 / 0.95 — below the 0.95 bar on 755252 but sitting firmly in the signal Gaussian.

**User decision (2026-04-17):** adopt `peakgauss3_mean_bg_p1`. The coverage loss on 755252 (6.3 / 100 coreg cells) is accepted in exchange for a GFP+ fraction comparable to the spot subjects. The selection rule is otherwise unchanged — the same family (PeakGauss3 + lower-tail percentile) that won the density side in subgoal 01, just with p = 1 instead of p = 0.1 to compensate for the broader signal component on 767022.

## 6. GFP+ fraction before / after

Baseline was "no threshold applied" (every HCR cell returned as GFP+, 100 %). v2.2 shrinks to 37 % / 19 %. Coverage annotations show the fraction of coreg-table HCR IDs retained (0.933 / 0.946 — below the 0.95 target on 755252, accepted). Downstream R1 origin-error impact is out of scope for this session and handled separately.

In [ ]:
from IPython.display import Image
Image(filename='../sessions/04_R1_coarse_align/subgoal_02_figures/gfp_frac_comparison.png')

### 6a. Final-protocol fit — `peakgauss3_mean_bg_p1`

Per-subject `log10(mean − background)` histogram for cells with `mean > bg` (grey = all HCR; blue = coreg-table subset on the right axis). The red solid curve is the rightmost component of the 3-component GMM fit. The red vertical line is the chosen threshold `exp(μ + z_{0.01} σ)` — lower-tail cutoff where 1 % of the fitted signal Gaussian sits below. `p = 1` (rather than the `p = 0.1` used on the spot side) compensates for the broader signal component on 767022.

In [ ]:
from IPython.display import Image
Image(filename='../sessions/04_R1_coarse_align/subgoal_02_figures/final_protocol_fit.png')

## 7. Summary

- **Chosen default:** `gfp_intensity_method='peakgauss3_mean_bg_p1'` in `benchmark_data_loader._load_gfp()`.
- **Rule is distribution-only** — no fixed-fraction knob — satisfying the generalisability constraint.
- **Coverage trade-off** accepted: cov_min = 0.933, gfp_frac_max = 0.366.
- Legacy `'none'` (no threshold) retained for rollback.
